# Assignment 2 — Cityscapes Semantic Segmentation (Single-file Colab Runner)
### DeepLabV3 (ResNet-50): Frozen backbone · Partial fine-tune (layer3+4) · Full fine-tune

This notebook is **self-contained**: all code from `src/` (data, transforms, metrics, models, train, evaluate) is embedded below — no `git clone`, no GitHub push.

**How to use**
1. Fill in the **CONFIGURATION PANEL** (Cityscapes username/password + hyperparameters).
2. **Runtime → Run all** (Ctrl+F9).
3. After the dataset is downloaded and prepared, the notebook **asks whether you want to resume training** — if yes, paste a Google Drive folder link (or a path like `MyDrive/cityscapes_runs`) that contains your previous `last.pt` / `best.pt` / `log.csv`.
4. At the end it **asks what to do with the results**: download them to your computer and/or save (overwrite) them to Google Drive.

**Drive folder layout** (written by the export step, read by the resume step):
```
<your Drive folder>/checkpoints/deeplabv3/local/                      ← Model 1 (frozen)
<your Drive folder>/checkpoints/deeplabv3/resnet_layer3_4_finetune/   ← Model 2 (partial)
<your Drive folder>/checkpoints/deeplabv3/resnet50_full_deeplabv3/    ← Model 3 (full)
      each containing: last.pt, best.pt, log.csv, metrics_test.json
```

In [33]:
# @title CONFIGURATION PANEL
# @markdown Fill this in, then Runtime → Run all.

# @markdown ---
# @markdown ### 1. Cityscapes account (register free at https://www.cityscapes-dataset.com/register/)
CITYSCAPES_USERNAME = ""  # @param {type:"string"}
CITYSCAPES_PASSWORD = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 2. Speed-up
FAST_PRE_RESIZE = True  # @param {type:"boolean"}
# @markdown Pre-resize the whole dataset to 512x1024 once (multi-process). Removes the CPU bottleneck (~78s → ~12-15s/epoch on A100).

# @markdown ---
# @markdown ### 3. Which models to train
TRAIN_MODEL_1_FROZEN = True  # @param {type:"boolean"}
TRAIN_MODEL_2_PARTIAL = True  # @param {type:"boolean"}
TRAIN_MODEL_3_FULL = True  # @param {type:"boolean"}

# @markdown ---
# @markdown ### 4. Hyperparameters
EPOCHS = 60  # @param {type:"integer"}
# @markdown When resuming, `EPOCHS` is the **total** target: a checkpoint at epoch 25 with EPOCHS=40 trains epochs 26 → 40.
RANDOM_SEED = 666  # @param {type:"integer"}
EVAL_EVERY = 1  # @param {type:"integer"}

# @markdown Batch size mode:
# @markdown - 'Auto (Detect GPU)': picks batch size from the detected GPU (A100 → 16).
# @markdown - 'Match Local (8)': batch 8, same as the local RTX 5070 Ti runs.
# @markdown - 'Custom': uses the two values below.
BATCH_SIZE_MODE = "Auto (Detect GPU)"  # @param ["Auto (Detect GPU)", "Match Local (8)", "Custom"]
CUSTOM_BATCH_SIZE = 8  # @param {type:"integer"}
CUSTOM_NUM_WORKERS = 4  # @param {type:"integer"}
USE_AMP = True  # @param {type:"boolean"}

# @markdown Learning rates:
HEAD_LR = 0.0001  # @param {type:"number"}
BACKBONE_LR_PARTIAL = 0.00001  # @param {type:"number"}
BACKBONE_LR_FULL = 0.000001  # @param {type:"number"}

# @markdown Dry-run (0 = full dataset; e.g. 20 for a quick test):
MAX_TRAIN_BATCHES = 0  # @param {type:"integer"}
MAX_EVAL_BATCHES = 0  # @param {type:"integer"}

print("=" * 75)
print(" CONFIGURATION:")
print(f"   • Cityscapes account   : {CITYSCAPES_USERNAME if CITYSCAPES_USERNAME else 'NOT SET'}")
print(f"   • Fast pre-resize      : {FAST_PRE_RESIZE}")
print(f"   • Train models         : M1={TRAIN_MODEL_1_FROZEN}  M2={TRAIN_MODEL_2_PARTIAL}  M3={TRAIN_MODEL_3_FULL}")
print(f"   • Epochs (total)       : {EPOCHS}")
print(f"   • Batch size mode      : {BATCH_SIZE_MODE} (custom={CUSTOM_BATCH_SIZE}, workers={CUSTOM_NUM_WORKERS})")
print(f"   • LR head / partial / full backbone: {HEAD_LR} / {BACKBONE_LR_PARTIAL} / {BACKBONE_LR_FULL}")
print(f"   • AMP                  : {USE_AMP}")
print(f"   • Seed                 : {RANDOM_SEED}")
print(f"   • Batch limit          : {'full dataset' if MAX_TRAIN_BATCHES == 0 else f'{MAX_TRAIN_BATCHES} (dry-run)'}")
print("=" * 75)

if not CITYSCAPES_USERNAME or not CITYSCAPES_PASSWORD:
    print("\n CITYSCAPES_USERNAME / CITYSCAPES_PASSWORD are empty — the download step will fail unless the dataset is already present.")

 CONFIGURATION:
   • Cityscapes account   : phu.dang296@hcmut.edu.vn
   • Fast pre-resize      : True
   • Train models         : M1=True  M2=True  M3=True
   • Epochs (total)       : 60
   • Batch size mode      : Auto (Detect GPU) (custom=8, workers=4)
   • LR head / partial / full backbone: 0.0001 / 1e-05 / 1e-06
   • AMP                  : True
   • Seed                 : 666
   • Batch limit          : full dataset


## Step 1: GPU check & install packages

In [2]:
# @title Check GPU & install dependencies
!nvidia-smi

import torch
print(f"\nPyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name        : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

# Keep Colab's default CUDA PyTorch
!pip install -q cityscapesscripts appdirs requests tqdm gdown
print("Environment ready")

Sat Nov  7 04:24:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   25C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Step 2: Embedded project code (from `src/`)
`common.py`, `cost.py`, `transforms.py`, `data.py`, `metrics.py`, `models/*`, `evaluate.py`, `train.py` merged into one cell, plus small helpers for Google Drive and the interactive prompts.

In [3]:
# @title Project source code (src/ embedded) + helpers
import csv
import json
import os
import random
import re
import shutil
import time
from pathlib import Path
from typing import Any, Dict

import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet50_Weights
from torchvision.models.segmentation import deeplabv3_resnet50

WORK_DIR = Path("/content/assignment2")
DATASET_DIR = WORK_DIR / "dataset"
SPLITS_DIR = WORK_DIR / "splits"
CKPT_ROOT = WORK_DIR / "checkpoints"
for _d in (DATASET_DIR, SPLITS_DIR, CKPT_ROOT):
    _d.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

NUM_CLASSES = 19
IGNORE_INDEX = 255

# Experiment registry: key -> (label, model name, checkpoint sub-dir)
RUNS = {
    "model1": {"label": "Model 1 (Frozen Backbone)","model": "deeplabv3_resnet50","run": "deeplabv3/local"},
    "model2": {"label": "Model 2 (Partial Fine-Tuning)","model": "resnet_layer3_4_finetune", "run": "deeplabv3/resnet_layer3_4_finetune"},
    "model3": {"label": "Model 3 (Full Fine-Tuning)","model": "resnet50_full_deeplabv3","run": "deeplabv3/resnet50_full_deeplabv3"},
}


# ============================== src/common.py ==============================
def select_loader(loaders, split="val"):
    """Works with get_loaders(cfg) returning a dict or a tuple."""
    positions = {"train": 0, "val": 1, "test": 2}
    idx = positions[split]
    if len(loaders) <= idx:
        raise ValueError(f"get_loaders(cfg) did not return a {split!r} loader")
    return loaders[idx]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_logits(output): #cuz deeplab v3 is a dict but Unet is a tensor
    return output["out"] if isinstance(output, dict) else output


# ============================== src/cost.py ==============================
class TrainingTimer:
    def __init__(self, device):
        self.device = device
        self.time = 0
        self.device_name = torch.cuda.get_device_name(self.device) if self.device.type == "cuda" else "cpu"

    def __enter__(self):
        if self.device.type == "cuda":
            torch.cuda.synchronize(self.device)
        self._start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc, tb):
        if self.device.type == "cuda":
            torch.cuda.synchronize(self.device)
        self._end = time.perf_counter()
        self.time = self._end - self._start


# ============================== src/transforms.py ==============================
class SegTransform:
    def __init__(self, is_train=True, base_size=(512, 1024), crop_size=(512, 512)):
        self.is_train = is_train
        self.base_size = base_size
        self.crop_size = crop_size
        self.color_jitter = transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def __call__(self, img, mask):
        target_w, target_h = self.base_size[1], self.base_size[0]
        if img.size != (target_w, target_h):
            img = TF.resize(img, self.base_size, interpolation=TF.InterpolationMode.BILINEAR)
        if mask.size != (target_w, target_h):
            mask = TF.resize(mask, self.base_size, interpolation=TF.InterpolationMode.NEAREST)

        if self.is_train:
            i, j, h, w = transforms.RandomCrop.get_params(img, output_size=self.crop_size)
            img = TF.crop(img, i, j, h, w)
            mask = TF.crop(mask, i, j, h, w)

            if random.random() > 0.5:
                img = TF.hflip(img)
                mask = TF.hflip(mask)

            img = self.color_jitter(img)

        img = TF.to_tensor(img)
        img = self.normalize(img)
        mask = torch.as_tensor(transforms.PILToTensor()(mask).squeeze(0), dtype=torch.int64)
        return img, mask


# ============================== src/data.py ==============================
class CityscapesSeg(Dataset):
    def __init__(self, data_dir, split_file, is_train=True):
        self.data_dir = Path(data_dir)
        self.transform = SegTransform(is_train=is_train)

        with open(split_file, "r") as f:
            self.samples = [line.strip() for line in f if line.strip()]

        # test.txt = official Cityscapes val; train.txt / val.txt = official train (city split)
        is_official_val = "test.txt" in str(split_file)
        split_folder = "val" if is_official_val else "train"

        self.img_dir = self.data_dir / "leftImg8bit_trainvaltest" / "leftImg8bit" / split_folder
        self.gt_dir = self.data_dir / "gtFine_trainvaltest" / "gtFine" / split_folder

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        city, stem_id = item.split("/")
        img_path = self.img_dir / city / f"{stem_id}_leftImg8bit.png"
        mask_path = self.gt_dir / city / f"{stem_id}_gtFine_trainIds.png"

        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)
        return self.transform(img, mask)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def get_loaders(cfg):
    """Returns train_loader, val_loader, test_loader.
    Images: float32 [B, 3, H, W] ImageNet-normalized. Masks: int64 [B, H, W] in 0..18 and 255.
    """
    data_dir = Path(cfg.get("data_dir", DATASET_DIR))
    splits_dir = Path(cfg.get("splits_dir", SPLITS_DIR))
    batch_size = cfg.get("batch_size", 4)
    num_workers = cfg.get("num_workers", 2)
    pin_memory = cfg.get("pin_memory", torch.cuda.is_available())
    seed = cfg.get("seed", 42)

    g = torch.Generator()
    g.manual_seed(seed)

    train_ds = CityscapesSeg(data_dir, splits_dir / "train.txt", is_train=True)
    val_ds = CityscapesSeg(data_dir, splits_dir / "val.txt", is_train=False)
    test_ds = CityscapesSeg(data_dir, splits_dir / "test.txt", is_train=False)

    persistent_workers = num_workers > 0
    prefetch_factor = 2 if num_workers > 0 else None

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=pin_memory,
        worker_init_fn=seed_worker, generator=g,
        drop_last=True, persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor,
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory,
        prefetch_factor=prefetch_factor,
    )
    return train_loader, val_loader, test_loader


# ============================== src/metrics.py ==============================
class ConfusionMatrix:
    """Confusion matrix accumulated on GPU/CPU (ignoring ignore_index=255).
    update(pred, target): pred [B, H, W] (or logits [B, C, H, W]), target [B, H, W]
    compute() -> dict(iou[19], miou, dice[19], mdice, pixel_acc)
    """

    def __init__(self, num_classes: int = 19, ignore_index: int = 255, device=None):
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.device = device
        self.total_mat = None

    def reset(self):
        self.total_mat = None

    @torch.no_grad()
    def update(self, pred: torch.Tensor, target: torch.Tensor):
        if pred.shape != target.shape:
            if pred.dim() == 4 and pred.shape[1] == self.num_classes:
                pred = torch.argmax(pred, dim=1)
            else:
                raise ValueError(f"Shape mismatch: pred {pred.shape} vs target {target.shape}")

        if self.total_mat is None:
            dev = self.device if self.device is not None else pred.device
            self.total_mat = torch.zeros((self.num_classes, self.num_classes), dtype=torch.int64, device=dev)

        pred = pred.to(self.total_mat.device)
        target = target.to(self.total_mat.device)

        mask = (target != self.ignore_index) & (target >= 0) & (target < self.num_classes)
        t = target[mask]
        p = pred[mask]

        indices = t * self.num_classes + p
        bins = torch.bincount(indices, minlength=self.num_classes ** 2)
        self.total_mat += bins.view(self.num_classes, self.num_classes)

    def compute(self) -> Dict[str, Any]:
        if self.total_mat is None:
            self.total_mat = torch.zeros((self.num_classes, self.num_classes), dtype=torch.int64)

        mat = self.total_mat.float()
        tp = torch.diag(mat)
        fp = mat.sum(dim=0) - tp
        fn = mat.sum(dim=1) - tp

        union = tp + fp + fn
        iou = torch.full((self.num_classes,), float("nan"), dtype=torch.float32, device=mat.device)
        valid_iou = union > 0
        iou[valid_iou] = tp[valid_iou] / union[valid_iou]

        dice_denom = 2 * tp + fp + fn
        dice = torch.full((self.num_classes,), float("nan"), dtype=torch.float32, device=mat.device)
        valid_dice = dice_denom > 0
        dice[valid_dice] = (2 * tp[valid_dice]) / dice_denom[valid_dice]

        miou = torch.nanmean(iou).item()
        mdice = torch.nanmean(dice).item()

        total_valid = mat.sum().item()
        pixel_acc = (tp.sum().item() / total_valid) if total_valid > 0 else 0.0

        return {
            "iou": iou.tolist(),
            "miou": miou,
            "dice": dice.tolist(),
            "mdice": mdice,
            "pixel_acc": pixel_acc,
            "confusion_matrix": self.total_mat.cpu().numpy(),
        }


# ============================== src/models/* ==============================
def _deeplab_r50():
    return deeplabv3_resnet50(
        weights=None, #no pretrained weights
        weights_backbone=ResNet50_Weights.IMAGENET1K_V1,
        #no head so its randomly initialized, but backbone is pretrained on ImageNet
        num_classes=19,
        aux_loss=False,
    )


def resnet_deeplabv3(cfg):
    """Model 1: backbone frozen, only the DeepLabV3 head is trained."""
    model = _deeplab_r50()
    for param in model.classifier.parameters():
        param.requires_grad = True
    for param in model.backbone.parameters():
        param.requires_grad = False
    return model


def resnet_layer3_4_finetune(cfg):
    """Model 2: layer1/2 frozen, layer3/4 + head trained."""
    model = _deeplab_r50()
    for param in model.classifier.parameters():
        param.requires_grad = True
    for param in model.backbone.parameters():
        param.requires_grad = False
    for param in model.backbone.layer3.parameters():
        param.requires_grad = True
    for param in model.backbone.layer4.parameters():
        param.requires_grad = True
    return model


def resnet50_full_deeplabv3(cfg):
    """Model 3: nothing frozen."""
    model = _deeplab_r50()
    for param in model.classifier.parameters():
        param.requires_grad = True
    for param in model.backbone.parameters():
        param.requires_grad = True
    return model


class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 19, base_channels: int = 32):
        super().__init__()
        c = base_channels
        self.inc = DoubleConv(in_channels, c)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(c, c * 2))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(c * 2, c * 4))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(c * 4, c * 8))
        self.bottleneck = nn.Sequential(nn.MaxPool2d(2), DoubleConv(c * 8, c * 16))
        self.up1 = nn.ConvTranspose2d(c * 16, c * 8, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(c * 16, c * 8)
        self.up2 = nn.ConvTranspose2d(c * 8, c * 4, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(c * 8, c * 4)
        self.up3 = nn.ConvTranspose2d(c * 4, c * 2, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(c * 4, c * 2)
        self.up4 = nn.ConvTranspose2d(c * 2, c, kernel_size=2, stride=2)
        self.conv_up4 = DoubleConv(c * 2, c)
        self.outc = nn.Conv2d(c, out_channels, kernel_size=1)
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        b = self.bottleneck(x4)
        d1 = self.conv_up1(torch.cat([self.up1(b), x4], dim=1))
        d2 = self.conv_up2(torch.cat([self.up2(d1), x3], dim=1))
        d3 = self.conv_up3(torch.cat([self.up3(d2), x2], dim=1))
        d4 = self.conv_up4(torch.cat([self.up4(d3), x1], dim=1))
        return self.outc(d4)


def build_unet(cfg):
    return UNet(in_channels=cfg.get("in_channels", 3), out_channels=cfg.get("num_classes", 19),
                base_channels=cfg.get("base_channels", 32))


def build_model(cfg):
    name = cfg.get("model")
    if name == "deeplabv3_resnet50":
        return resnet_deeplabv3(cfg)
    if name == "resnet_layer3_4_finetune":
        return resnet_layer3_4_finetune(cfg)
    if name == "resnet50_full_deeplabv3":
        return resnet50_full_deeplabv3(cfg)
    if name == "unet":
        return build_unet(cfg)
    raise ValueError(f"Unknown model {name!r}.")


# ============================== src/evaluate.py ==============================
def load_checkpoint(path, model):
    checkpoint = torch.load(Path(path), map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["model"])
    return checkpoint


@torch.inference_mode()
def validate(model, loader, device, amp=False, max_batches=0, metric=None):
    if max_batches < 0:
        raise ValueError("max_batches must be non-negative; use 0 for all batches")
    model.eval()
    if metric is None:
        metric = ConfusionMatrix(num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX, device=device)
    metric.reset()
    criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
    use_amp = amp and device.type == "cuda"
    total_samples = 0
    total_loss = 0
    total_batches = min(len(loader), max_batches) if max_batches else len(loader)

    for i, (images, masks) in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, dtype=torch.long, non_blocking=True)

        if not (masks != IGNORE_INDEX).any().item():
            continue

        with torch.autocast(device_type=device.type, enabled=use_amp):
            logits = get_logits(model(images))
            loss = criterion(logits, masks)

        total_loss += loss.item() * images.size(0)
        total_samples += images.size(0)
        metric.update(logits, masks)

        if (i + 1) % 50 == 0 or (i + 1) == total_batches:
            print(f"  Batch [{i + 1}/{total_batches}] - Eval_loss_current_batch: {loss.item():.4f}")

    if total_samples == 0:
        raise ValueError("No valid labeled pixels in evaluation data")

    scores = metric.compute()
    return {
        "avr_loss": total_loss / total_samples,
        "total_samples": total_samples,
        "miou": scores["miou"],
        "mdice": scores["mdice"],
        "pixel_acc": scores["pixel_acc"],
        "per_class_iou": scores["iou"],
    }


def evaluate_checkpoint(cfg, checkpoint_path, split="test", max_batches=0, save_json=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(cfg).to(device)
    checkpoint = load_checkpoint(checkpoint_path, model)
    print("Checkpoint:", checkpoint_path)

    loader = select_loader(get_loaders(cfg), split)
    with TrainingTimer(device) as eval_time:
        result = validate(model, loader, device, amp=bool(cfg.get("amp", False)), max_batches=max_batches)
    record = {
        "epoch": checkpoint["epoch"],
        "split": split,
        "device_name": eval_time.device_name,
        "eval_seconds": eval_time.time,
        "eval_size": result["total_samples"],
        "eval_loss": result["avr_loss"],
        "eval_miou": result["miou"],
        "eval_dice": result["mdice"],
        "pixel_acc": result["pixel_acc"],
    }
    print(" | ".join(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}" for k, v in record.items()))
    if max_batches:
        print(f"Limited evaluation: at most {max_batches} batches.")

    if save_json:
        save_path = Path(save_json)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump({**result, **record}, f, indent=4)
        print("Metrics saved to:", save_path)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {**result, **record}


# ============================== src/train.py ==============================
def train_one_epoch(model, loader, optimizer, device, max_batches=0, amp=False, scaler=None):
    model.train()
    backbone = getattr(model, "backbone", None)
    if backbone and not any(p.requires_grad for p in backbone.parameters()):
        backbone.eval()  # keep frozen BatchNorm statistics fixed
    criterion = nn.CrossEntropyLoss(ignore_index=255)
    total_samples = 0
    total_loss = 0
    use_amp = amp and device.type == "cuda"

    for i, (images, masks) in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        if not (masks != 255).any().item():
            continue
        if images.size(0) <= 1:
            continue

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, dtype=torch.long, non_blocking=True)
        optimizer.zero_grad()

        with torch.autocast(device_type=device.type, enabled=use_amp):
            logits = get_logits(model(images))
            loss = criterion(logits, masks)

        if use_amp and scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_samples += images.size(0)

        if (i + 1) % 50 == 0 or (i + 1) == max_batches:
            print(f"  Batch [{i + 1}/{max_batches}] - Avr_loss_current_batches: {loss.item():.4f}")

    if total_samples == 0:
        raise ValueError("No valid training samples were processed")
    return {"avr_loss": total_loss / total_samples, "total_samples": total_samples}


def save_checkpoint(path, epoch, model, optimizer, best_miou, scaler=None): #only on colab local disk
    state = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "best_miou": best_miou,
    }
    if scaler is not None and scaler.is_enabled():
        state["scaler"] = scaler.state_dict()
    tmp = Path(str(path) + ".tmp")
    torch.save(state, tmp)
    os.replace(tmp, path)  # atomic: never leaves a half-written last.pt


def build_optimizer(cfg, model):
    backbone_lr = cfg.get("backbone_lr")
    if backbone_lr is None:
        return torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=float(cfg["lr"]))
    return torch.optim.AdamW([
        {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": float(backbone_lr)},
        {"params": [p for p in model.classifier.parameters() if p.requires_grad], "lr": float(cfg["lr"])},
    ])


def train_model(cfg):
    """Train one experiment. If cfg['resume'] points to a checkpoint, continue from it
    until cfg['epochs'] total epochs are reached."""
    out_dir = Path(cfg["checkpoint_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)
    log_path = out_dir / "log.csv"

    seed = int(cfg.get("seed", 42))
    set_seed(seed)

    loaders = get_loaders(cfg)
    train_loader = select_loader(loaders, "train")
    eval_loader = select_loader(loaders, "val")
    total_epochs = int(cfg["epochs"])
    eval_every = max(1, int(cfg.get("eval_every", 1)))
    max_train_batches = len(train_loader) if cfg.get("max_train_batches", 0) == 0 else min(len(train_loader), int(cfg["max_train_batches"]))
    max_eval_batches = len(eval_loader) if cfg.get("max_eval_batches", 0) == 0 else min(len(eval_loader), int(cfg["max_eval_batches"]))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    model = build_model(cfg).to(device)
    optimizer = build_optimizer(cfg, model)
    use_amp = bool(cfg.get("amp", False)) and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    print("Device:", device)

    start_epoch = 1
    best_miou = -1.0
    resume_path = cfg.get("resume")
    if resume_path and Path(resume_path).is_file():
        checkpoint = torch.load(resume_path, map_location="cpu", weights_only=True)
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        # The optimizer LR comes from the checkpoint; re-apply the LRs from the config panel
        for group, lr in zip(optimizer.param_groups, build_optimizer(cfg, model).param_groups):
            group["lr"] = lr["lr"]
        if use_amp and "scaler" in checkpoint:
            scaler.load_state_dict(checkpoint["scaler"])
        start_epoch = int(checkpoint["epoch"]) + 1
        best_miou = float(checkpoint.get("best_miou", -1.0))
        print(f"Resumed from {resume_path}: epoch {checkpoint['epoch']} done, best mIoU so far {best_miou:.4f}")

    if start_epoch > total_epochs:
        print(f"Already trained {start_epoch - 1}/{total_epochs} epochs — nothing to do. Increase EPOCHS to train more.")
        del model, optimizer
        return

    for epoch in range(start_epoch, total_epochs + 1):
        print(f"Epochs {epoch}/{total_epochs} :")
        with TrainingTimer(device) as train_time:
            train_result = train_one_epoch(model, train_loader, optimizer, device,
                                           max_batches=max_train_batches, amp=use_amp, scaler=scaler)

        do_eval = (epoch % eval_every == 0) or epoch == total_epochs
        improved = False
        eval_result, eval_seconds = None, 0.0
        if do_eval:
            metric = ConfusionMatrix(device=device)
            with TrainingTimer(device) as eval_time:
                eval_result = validate(model, eval_loader, device, amp=use_amp,
                                       max_batches=max_eval_batches, metric=metric)
            eval_seconds = eval_time.time
            improved = eval_result["miou"] > best_miou
            best_miou = max(best_miou, eval_result["miou"])

        save_checkpoint(out_dir / "last.pt", epoch, model, optimizer, best_miou, scaler)
        if improved:
            save_checkpoint(out_dir / "best.pt", epoch, model, optimizer, best_miou, scaler)

        er = eval_result or {}
        record = {
            "epoch": epoch,
            "device_name": train_time.device_name,
            "train_seconds": train_time.time,
            "train_size": train_result["total_samples"],
            "train_avr_loss": train_result["avr_loss"],
            "eval_seconds": eval_seconds if do_eval else "",
            "eval_size": er.get("total_samples", ""),
            "eval_loss": er.get("avr_loss", ""),
            "eval_miou": er.get("miou", ""),
            "eval_dice": er.get("mdice", ""),
            "eval_accuracy": er.get("pixel_acc", ""),
            "eval_per_class_iou": er.get("per_class_iou", ""),
        }
        new_file = not log_path.exists()
        with log_path.open("a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(record.keys()))
            if new_file:
                writer.writeheader()
            writer.writerow(record)

        print(" | ".join(
            f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}"
            for k, v in record.items() if k != "eval_per_class_iou"
        ) + ("  new best" if improved else ""))

    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================== Interactive + Google Drive helpers ==============================
DRIVE_MOUNT = Path("/content/drive")


def ask_yes_no(question, default=False):
    hint = "[Y/n]" if default else "[y/N]"
    while True:
        ans = input(f"{question} {hint}: ").strip().lower()
        if not ans:
            return default
        if ans in ("y", "yes", "c", "co", "có"):
            return True
        if ans in ("n", "no", "k", "khong", "không"):
            return False
        print("   Please answer y or n.")


def mount_drive():
    if not (DRIVE_MOUNT / "MyDrive").is_dir():
        from google.colab import drive
        drive.mount(str(DRIVE_MOUNT))
    return DRIVE_MOUNT / "MyDrive"


def _drive_id_of(path):
    try:
        return os.getxattr(str(path), "user.drive.id").decode()
    except Exception:
        return None


def _folder_id_from_link(text):
    m = re.search(r"/folders/([A-Za-z0-9_-]{10,})", text) or re.search(r"[?&]id=([A-Za-z0-9_-]{10,})", text)
    return m.group(1) if m else None


def resolve_drive_folder(text, create=False, max_depth=5):
    """Accepts a Google Drive folder link (https://drive.google.com/drive/folders/<ID>)
    or a path (MyDrive/xxx, /content/drive/MyDrive/xxx, or xxx relative to MyDrive).
    Mounts Drive and returns the local mounted path of that folder."""
    s = text.strip().strip('"').strip("'")
    if not s:
        raise ValueError("Empty input")
    my_drive = mount_drive()
    folder_id = _folder_id_from_link(s)

    if folder_id:
        # Shared folders added as a shortcut show up here
        shortcut = DRIVE_MOUNT / ".shortcut-targets-by-id" / folder_id
        if shortcut.is_dir():
            subs = [p for p in shortcut.iterdir() if p.is_dir()]
            return subs[0] if len(subs) == 1 else shortcut
        print(f"Looking for folder id {folder_id} in your mounted Drive (may take a moment)...")
        roots = [my_drive] + ([DRIVE_MOUNT / "Shareddrives"] if (DRIVE_MOUNT / "Shareddrives").is_dir() else [])
        for root_dir in roots:
            if _drive_id_of(root_dir) == folder_id:
                return root_dir
            for cur, dirs, _ in os.walk(root_dir):
                if len(Path(cur).relative_to(root_dir).parts) >= max_depth:
                    dirs[:] = []
                    continue
                for d in dirs:
                    p = Path(cur) / d
                    if _drive_id_of(p) == folder_id:
                        return p
        raise FileNotFoundError(
            "Could not find that folder in your mounted Drive. If it belongs to someone else, open it in Drive and "
            "use 'Organize → Add shortcut' to put it in My Drive, then paste the link again — or paste a path like "
            "MyDrive/cityscapes_runs instead.")

    if s.startswith("/content/drive"):
        p = Path(s)
    elif s.startswith("MyDrive") or s.startswith("Shareddrives"):
        p = DRIVE_MOUNT / s
    else:
        p = my_drive / s.lstrip("/")
    if not p.is_dir():
        if create:
            p.mkdir(parents=True, exist_ok=True)
        else:
            raise FileNotFoundError(f"Folder does not exist: {p}")
    return p


def find_run_dir(base, run_rel):
    """Find the folder holding last.pt/best.pt for a run inside a Drive folder."""
    base = Path(base)
    for c in (base / "checkpoints" / run_rel, base / run_rel, base / Path(run_rel).name):
        if (c / "last.pt").is_file() or (c / "best.pt").is_file():
            return c
    return None


print("Source code loaded. Working dir:", WORK_DIR)

Source code loaded. Working dir: /content/assignment2


## Step 3: Download Cityscapes from the web & unzip

In [4]:
# @title Download dataset (gtFine + leftImg8bit) and unzip
import appdirs
from cityscapesscripts.download import downloader

already_ready = (
    (DATASET_DIR / "leftImg8bit_trainvaltest" / "leftImg8bit").is_dir() and
    (DATASET_DIR / "gtFine_trainvaltest" / "gtFine").is_dir()
)

if already_ready:
    print("Dataset already present. Skipping download.")
else:
    if not (CITYSCAPES_USERNAME and CITYSCAPES_PASSWORD):
        raise ValueError("Fill CITYSCAPES_USERNAME and CITYSCAPES_PASSWORD in the CONFIGURATION PANEL and run again.")

    print(f"Logging in to Cityscapes as {CITYSCAPES_USERNAME}...")
    cred_dir = appdirs.user_data_dir("cityscapesscripts", "cityscapes")
    os.makedirs(cred_dir, exist_ok=True)
    with open(os.path.join(cred_dir, "credentials.json"), "w", encoding="utf-8") as f:
        json.dump({"username": CITYSCAPES_USERNAME, "password": CITYSCAPES_PASSWORD}, f)

    session = downloader.login()
    print("Logged in")

    packages = []
    if not (DATASET_DIR / "gtFine_trainvaltest.zip").exists() and not (DATASET_DIR / "gtFine_trainvaltest").is_dir():
        packages.append("gtFine_trainvaltest.zip")
    if not (DATASET_DIR / "leftImg8bit_trainvaltest.zip").exists() and not (DATASET_DIR / "leftImg8bit_trainvaltest").is_dir():
        packages.append("leftImg8bit_trainvaltest.zip")

    if packages:
        print(f"Downloading {packages} (resumable)...")
        downloader.download_packages(session=session, package_names=packages,
                                     destination_path=str(DATASET_DIR), resume=True)
        print("Download finished")

    print("\nUnzipping...")
    if not (DATASET_DIR / "gtFine_trainvaltest" / "gtFine").is_dir() and (DATASET_DIR / "gtFine_trainvaltest.zip").is_file():
        print("  • gtFine_trainvaltest.zip (252 MB)...")
        !unzip -q -o "{DATASET_DIR}/gtFine_trainvaltest.zip" -d "{DATASET_DIR}/gtFine_trainvaltest"
    if not (DATASET_DIR / "leftImg8bit_trainvaltest" / "leftImg8bit").is_dir() and (DATASET_DIR / "leftImg8bit_trainvaltest.zip").is_file():
        print("  • leftImg8bit_trainvaltest.zip (11.5 GB)...")
        !unzip -q -o "{DATASET_DIR}/leftImg8bit_trainvaltest.zip" -d "{DATASET_DIR}/leftImg8bit_trainvaltest"

    # Free disk space once extracted
    for z in ("gtFine_trainvaltest.zip", "leftImg8bit_trainvaltest.zip"):
        if (DATASET_DIR / z).is_file():
            (DATASET_DIR / z).unlink()
    print("Dataset extracted")

Logging in to Cityscapes as phu.dang296@hcmut.edu.vn...
Logged in


Download progress: 100%|█████████▉| 241M/241M [00:11<00:00, 22.9MB/s]


Download progress:  98%|█████████▊| 10.8G/11.0G [12:35<00:14, 15.3MB/s]


Download finished

Unzipping...
  • gtFine_trainvaltest.zip (252 MB)...
  • leftImg8bit_trainvaltest.zip (11.5 GB)...
Dataset extracted


## Step 4: Prepare data — trainIds masks (34 → 19 classes) & city-based splits
Embedded versions of `scripts/prepare_data.py` and `scripts/create_splits.py`.

In [5]:
# @title Generate *_gtFine_trainIds.png and splits/train|val|test.txt
import concurrent.futures
from cityscapesscripts.helpers.labels import labels as CS_LABELS
from tqdm import tqdm

# ---- prepare_data.py ----
#make a lookup table to convert labelIds -> trainIds # [0,1,2,3,4,5,...,255] -> [255,1..2....] 
LUT = np.full(256, 255, dtype=np.uint8)
for label in CS_LABELS:
    if label.id >= 0 and 0 <= label.trainId < 19:
        LUT[label.id] = label.trainId



def _to_train_ids(label_path):
    #label_path = ".../train/aachen/aachen_000000_000019_gtFine_labelIds.png"
    #out        = ".../train/aachen/aachen_000000_000019_gtFine_trainIds.png"
    out = label_path.replace("labelIds", "trainIds")
    if os.path.isfile(out):
        return
    Image.fromarray(LUT[np.array(Image.open(label_path))]).save(out)


label_files = [str(p) for p in (DATASET_DIR / "gtFine_trainvaltest").rglob("*_gtFine_labelIds.png")]
if not label_files:
    raise RuntimeError(f"No *_gtFine_labelIds.png found under {DATASET_DIR}. Did the download/unzip step succeed?")

todo = [p for p in label_files if not os.path.isfile(p.replace("labelIds", "trainIds"))]
if todo:
    print(f"Creating {len(todo)} trainIds masks...")
    with concurrent.futures.ProcessPoolExecutor(max_workers=os.cpu_count() or 4) as ex: #multi process
        list(tqdm(ex.map(_to_train_ids, todo, chunksize=32), total=len(todo), desc="trainIds"))
n_masks = len(list(DATASET_DIR.rglob("*_gtFine_trainIds.png")))
print(f"{n_masks} trainIds masks ready")

# ---- create_splits.py ----
TRAIN_GT_DIR = DATASET_DIR / "gtFine_trainvaltest" / "gtFine" / "train"
VAL_GT_DIR = DATASET_DIR / "gtFine_trainvaltest" / "gtFine" / "val"
VAL_CITIES = {"erfurt", "jena", "tubingen", "zurich"}


def get_city_samples(city_dir):
    return [f"{city_dir.name}/{f.name.replace('_gtFine_labelIds.png', '')}"
            for f in sorted(city_dir.glob("*_gtFine_labelIds.png"))]


train_samples, val_samples, test_samples = [], [], []
train_cities, test_cities = set(), set()
for c_dir in sorted(d for d in TRAIN_GT_DIR.iterdir() if d.is_dir()):
    if c_dir.name in VAL_CITIES:
        val_samples.extend(get_city_samples(c_dir))
    else:
        train_cities.add(c_dir.name)
        train_samples.extend(get_city_samples(c_dir))
for c_dir in sorted(d for d in VAL_GT_DIR.iterdir() if d.is_dir()):
    test_cities.add(c_dir.name)
    test_samples.extend(get_city_samples(c_dir))

assert not train_cities & VAL_CITIES and not train_cities & test_cities and not VAL_CITIES & test_cities, "City leakage!"
assert not set(train_samples) & set(val_samples) and not set(train_samples) & set(test_samples), "Duplicate samples!"

for name, items in (("train.txt", train_samples), ("val.txt", val_samples), ("test.txt", test_samples)):
    (SPLITS_DIR / name).write_text("\n".join(items) + "\n")

print("DATASET SPLIT STATISTICS")
print(f"  Train: {len(train_samples)} images from {len(train_cities)} cities")
print(f"  Val  : {len(val_samples)} images from {sorted(VAL_CITIES)}")
print(f"  Test : {len(test_samples)} images from {len(test_cities)} cities (official val)")

Creating 5000 trainIds masks...


trainIds: 100%|██████████| 5000/5000 [00:03<00:00, 1630.15it/s]

5000 trainIds masks ready
DATASET SPLIT STATISTICS
  Train: 2481 images from 14 cities
  Val  : 494 images from ['erfurt', 'jena', 'tubingen', 'zurich']
  Test : 500 images from 3 cities (official val)


## Step 4.1: Fast parallel pre-resize to 512×1024
Resizes every RGB image (bilinear) and trainIds mask (nearest) once, so training never decodes 2048×1024 PNGs.

In [6]:
# @title Pre-resize dataset (multi-process)
TARGET_W, TARGET_H = 1024, 512


def _resize_file(args):
    path_str, is_mask = args
    try:
        with Image.open(path_str) as im:
            if im.size == (TARGET_W, TARGET_H):
                return False
            resized = im.resize((TARGET_W, TARGET_H), Image.Resampling.NEAREST if is_mask else Image.Resampling.BILINEAR)
        resized.save(path_str)
        return True
    except Exception:
        return False


if not FAST_PRE_RESIZE:
    print("if FAST_PRE_RESIZE = False — using original 2048x1024 images.")
else:
    img_files = list(DATASET_DIR.rglob("*_leftImg8bit.png"))
    mask_files = list(DATASET_DIR.rglob("*_gtFine_trainIds.png"))
    print(f"{len(img_files)} RGB images, {len(mask_files)} trainIds masks")

    need_resize = False
    for sample in (img_files[:1] + img_files[-1:] + mask_files[-1:]):
        with Image.open(sample) as im:
            if im.size != (TARGET_W, TARGET_H):
                need_resize = True

    if not need_resize:
        print("Dataset already pre-resized to 512x1024.")
    else:
        jobs = [(str(p), False) for p in img_files] + [(str(p), True) for p in mask_files]
        with concurrent.futures.ProcessPoolExecutor(max_workers=os.cpu_count() or 4) as ex:
            list(tqdm(ex.map(_resize_file, jobs, chunksize=16), total=len(jobs), desc="Resize"))
        print("Pre-resize done")

5000 RGB images, 5000 trainIds masks


Resize: 100%|██████████| 10000/10000 [00:27<00:00, 363.62it/s]

Pre-resize done


## Step 5: Resume training?
You will be asked whether to resume. If **yes**, paste a Google Drive **folder link** (e.g. `https://drive.google.com/drive/folders/1AbC...`) or a path such as `MyDrive/cityscapes_runs`. Drive gets mounted and, for each model, `last.pt` (or `best.pt`), `best.pt`, `log.csv`, `metrics_test.json` are copied into the local `checkpoints/` folder.

In [34]:
# @title Ask: resume training from Google Drive?
RESUME_FROM = {}          # key -> local checkpoint path to resume from
DRIVE_FOLDER = None       # remembered for the export step at the end


def _ckpt_info(path):
    try:
        ck = torch.load(path, map_location="cpu", weights_only=True)
        return int(ck.get("epoch", 0)), float(ck.get("best_miou", -1))
    except Exception as e:
        print(f"   Could not read {path}: {e}")
        return None, None


# Checkpoints already in this Colab session (e.g. a cell was re-run)?
local_existing = {k: CKPT_ROOT / r["run"] / "last.pt" for k, r in RUNS.items() if (CKPT_ROOT / r["run"] / "last.pt").is_file()}

if ask_yes_no("Do you want to RESUME training from checkpoints on Google Drive?"):
    while DRIVE_FOLDER is None:
        link = input("Paste the Google Drive folder link (or a path like MyDrive/cityscapes_runs): ")
        try:
            DRIVE_FOLDER = resolve_drive_folder(link)
            print(f"Using Drive folder: {DRIVE_FOLDER}")
        except Exception as e:
            print(f"{e}")
            if not ask_yes_no("Try another link?", default=True):
                break

    if DRIVE_FOLDER is not None:
        for key, run in RUNS.items():
            src = find_run_dir(DRIVE_FOLDER, run["run"])
            if src is None:
                print(f"  • {run['label']}: no last.pt/best.pt found in Drive → will train from scratch")
                continue
            dst = CKPT_ROOT / run["run"]
            dst.mkdir(parents=True, exist_ok=True)
            for name in ("last.pt", "best.pt", "log.csv", "metrics_test.json"):
                if (src / name).is_file():
                    shutil.copy2(src / name, dst / name)
            ck = dst / "last.pt" if (dst / "last.pt").is_file() else dst / "best.pt"
            epoch, best = _ckpt_info(ck)
            if epoch is None:
                continue
            RESUME_FROM[key] = str(ck)
            print(f"  {run['label']}: {ck.name} from {src} → epoch {epoch}, best val mIoU {best:.4f}")
elif local_existing and ask_yes_no(f"Found checkpoints from this Colab session for {list(local_existing)}. Resume from those?", default=True):
    for key, ck in local_existing.items():
        epoch, best = _ckpt_info(ck)
        if epoch is not None:
            RESUME_FROM[key] = str(ck)
            print(f"  {RUNS[key]['label']}: local last.pt → epoch {epoch}, best val mIoU {best:.4f}")
else:
    # Fresh start: clear stale local logs/checkpoints so log.csv is not appended to an old run
    for run in RUNS.values():
        d = CKPT_ROOT / run["run"]
        for name in ("last.pt", "best.pt", "log.csv", "metrics_test.json"):
            if (d / name).is_file():
                (d / name).unlink()
    print("Training from scratch.")

print("\nResume plan:", {RUNS[k]["label"]: v for k, v in RESUME_FROM.items()} or "none (fresh training)")

Do you want to RESUME training from checkpoints on Google Drive? [y/N]: y
Paste the Google Drive folder link (or a path like MyDrive/cityscapes_runs): https://drive.google.com/drive/folders/1kFyCRTMoaTPIKx3f1SnrZEK_HweD8RPG?hl=vi
Using Drive folder: /content/drive/.shortcut-targets-by-id/1kFyCRTMoaTPIKx3f1SnrZEK_HweD8RPG/assignment2
  Model 1 (Frozen Backbone): last.pt from /content/drive/.shortcut-targets-by-id/1kFyCRTMoaTPIKx3f1SnrZEK_HweD8RPG/assignment2/checkpoints/deeplabv3/local → epoch 40, best val mIoU 0.5940
  • Model 2 (Partial Fine-Tuning): no last.pt/best.pt found in Drive → will train from scratch
  • Model 3 (Full Fine-Tuning): no last.pt/best.pt found in Drive → will train from scratch

Resume plan: {'Model 1 (Frozen Backbone)': '/content/assignment2/checkpoints/deeplabv3/local/last.pt'}


## Step 6: Detect GPU & build the training configs

In [36]:
# @title Auto-tune batch size / workers for the detected GPU and build configs
cpu_cores = os.cpu_count() or 2

if not torch.cuda.is_available():
    detected_gpu_title, auto_batch_size, auto_workers = "CPU fallback (no CUDA)", 4, 2
else:
    raw_device_name = torch.cuda.get_device_name(0).upper()
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    if "A100" in raw_device_name:
        detected_gpu_title, auto_batch_size, auto_workers = f"NVIDIA A100 ({vram_gb:.1f} GB)", 16, min(cpu_cores, 8)
    elif "L4" in raw_device_name:
        detected_gpu_title, auto_batch_size, auto_workers = f"NVIDIA L4 ({vram_gb:.1f} GB)", 12, min(cpu_cores, 6)
    elif "T4" in raw_device_name:
        detected_gpu_title, auto_batch_size, auto_workers = f"NVIDIA T4 ({vram_gb:.1f} GB)", 8, min(cpu_cores, 4)
    elif "V100" in raw_device_name:
        detected_gpu_title, auto_batch_size, auto_workers = f"NVIDIA V100 ({vram_gb:.1f} GB)", 12, min(cpu_cores, 6)
    else:
        detected_gpu_title = f"{raw_device_name} ({vram_gb:.1f} GB)"
        auto_batch_size = 16 if vram_gb >= 30 else (12 if vram_gb >= 20 else (8 if vram_gb >= 12 else 4))
        auto_workers = min(cpu_cores, 6)

if BATCH_SIZE_MODE == "Auto (Detect GPU)":
    final_batch_size, final_workers = auto_batch_size, auto_workers
elif BATCH_SIZE_MODE == "Match Local (8)":
    final_batch_size, final_workers = 8, auto_workers
else:
    final_batch_size, final_workers = CUSTOM_BATCH_SIZE, CUSTOM_NUM_WORKERS

BASE_CFG = {
    "weights_backbone": "IMAGENET1K_V1",
    "num_classes": 19,
    "ignore_index": 255,
    "seed": RANDOM_SEED,
    "epochs": EPOCHS,
    "amp": USE_AMP,
    "batch_size": final_batch_size,
    "num_workers": final_workers,
    "pin_memory": True,
    "max_train_batches": MAX_TRAIN_BATCHES,
    "max_eval_batches": MAX_EVAL_BATCHES,
    "eval_every": EVAL_EVERY,
    "data_dir": str(DATASET_DIR),
    "splits_dir": str(SPLITS_DIR),
    "lr": HEAD_LR,
    "head_lr": HEAD_LR, 
}
CFGS = {
    "model1": {**BASE_CFG, "model": RUNS["model1"]["model"]},
    "model2": {**BASE_CFG, "model": RUNS["model2"]["model"], "backbone_lr": BACKBONE_LR_PARTIAL},
    "model3": {**BASE_CFG, "model": RUNS["model3"]["model"], "backbone_lr": BACKBONE_LR_FULL},
}
for key, cfg in CFGS.items():
    cfg["checkpoint_dir"] = str(CKPT_ROOT / RUNS[key]["run"])
    cfg["resume"] = RESUME_FROM.get(key)

print("=" * 75)
print(f"GPU: {detected_gpu_title} | CPU cores: {cpu_cores}")
print(f"   batch_size={final_batch_size}, num_workers={final_workers}, epochs(total)={EPOCHS}, amp={USE_AMP}")
for key, cfg in CFGS.items():
    print(f"   • {RUNS[key]['label']:32s} lr={cfg['lr']}  backbone_lr={cfg.get('backbone_lr')}  resume={'yes' if cfg['resume'] else 'no'}")
print("=" * 75)

GPU: NVIDIA RTX PRO 6000 BLACKWELL SERVER EDITION (95.0 GB) | CPU cores: 48
   batch_size=16, num_workers=6, epochs(total)=60, amp=True
   • Model 1 (Frozen Backbone)        lr=0.0001  backbone_lr=None  resume=yes
   • Model 2 (Partial Fine-Tuning)    lr=0.0001  backbone_lr=1e-05  resume=no
   • Model 3 (Full Fine-Tuning)       lr=0.0001  backbone_lr=1e-06  resume=no


## Step 7: Train the 3 DeepLabV3 models

In [37]:
# @title Model 1: Frozen Backbone (train ASPP head only)
if not TRAIN_MODEL_1_FROZEN:
    print(" Skipped (TRAIN_MODEL_1_FROZEN = False)")
else:
    print("Training {}...".format(RUNS["model1"]["label"]))
    train_model(CFGS["model1"])
    if (CKPT_ROOT / RUNS["model1"]["run"] / "best.pt").is_file():
        print("{} done → {}".format(RUNS["model1"]["label"], CKPT_ROOT / RUNS["model1"]["run"]))
    else:
        print("No best.pt produced — check the log above.")

Training Model 1 (Frozen Backbone)...
Device: cuda
Resumed from /content/assignment2/checkpoints/deeplabv3/local/last.pt: epoch 40 done, best mIoU so far 0.5940
Epochs 41/60 :
  Batch [50/155] - Avr_loss_current_batches: 0.1541
  Batch [100/155] - Avr_loss_current_batches: 0.1568
  Batch [150/155] - Avr_loss_current_batches: 0.1558
  Batch [155/155] - Avr_loss_current_batches: 0.1669
  Batch [31/31] - Eval_loss_current_batch: 0.2834
epoch: 41 | device_name: NVIDIA RTX PRO 6000 Blackwell Server Edition | train_seconds: 15.0548 | train_size: 2480 | train_avr_loss: 0.1485 | eval_seconds: 3.4384 | eval_size: 494 | eval_loss: 0.2768 | eval_miou: 0.5953 | eval_dice: 0.7275 | eval_accuracy: 0.9170  new best
Epochs 42/60 :
  Batch [50/155] - Avr_loss_current_batches: 0.1399
  Batch [100/155] - Avr_loss_current_batches: 0.1558
  Batch [150/155] - Avr_loss_current_batches: 0.1632
  Batch [155/155] - Avr_loss_current_batches: 0.1367
  Batch [31/31] - Eval_loss_current_batch: 0.2812
epoch: 42 | de

In [ ]:
# @title Model 2: Partial Fine-Tuning (layer3 + layer4 + head)
if not TRAIN_MODEL_2_PARTIAL:
    print(" Skipped (TRAIN_MODEL_2_PARTIAL = False)")
else:
    print("Training {}...".format(RUNS["model2"]["label"]))
    train_model(CFGS["model2"])
    if (CKPT_ROOT / RUNS["model2"]["run"] / "best.pt").is_file():
        print("{} done → {}".format(RUNS["model2"]["label"], CKPT_ROOT / RUNS["model2"]["run"]))
    else:
        print("No best.pt produced — check the log above.")

Training Model 2 (Partial Fine-Tuning)...
Device: cuda
Epochs 1/60 :
  Batch [50/155] - Avr_loss_current_batches: 0.6449
  Batch [100/155] - Avr_loss_current_batches: 0.5121
  Batch [150/155] - Avr_loss_current_batches: 0.4847
  Batch [155/155] - Avr_loss_current_batches: 0.5332
  Batch [31/31] - Eval_loss_current_batch: 0.5035
epoch: 1 | device_name: NVIDIA RTX PRO 6000 Blackwell Server Edition | train_seconds: 23.2526 | train_size: 2480 | train_avr_loss: 0.7036 | eval_seconds: 3.3531 | eval_size: 494 | eval_loss: 0.4654 | eval_miou: 0.3342 | eval_dice: 0.4018 | eval_accuracy: 0.8755  new best
Epochs 2/60 :
  Batch [50/155] - Avr_loss_current_batches: 0.3507
  Batch [100/155] - Avr_loss_current_batches: 0.3823
  Batch [150/155] - Avr_loss_current_batches: 0.3126
  Batch [155/155] - Avr_loss_current_batches: 0.2803
  Batch [31/31] - Eval_loss_current_batch: 0.4048
epoch: 2 | device_name: NVIDIA RTX PRO 6000 Blackwell Server Edition | train_seconds: 22.5511 | train_size: 2480 | train_av

In [ ]:
# @title Model 3: Full Fine-Tuning (differential LR)
if not TRAIN_MODEL_3_FULL:
    print(" Skipped (TRAIN_MODEL_3_FULL = False)")
else:
    print("Training {}...".format(RUNS["model3"]["label"]))
    train_model(CFGS["model3"])
    if (CKPT_ROOT / RUNS["model3"]["run"] / "best.pt").is_file():
        print("{} done → {}".format(RUNS["model3"]["label"], CKPT_ROOT / RUNS["model3"]["run"]))
    else:
        print("No best.pt produced — check the log above.")

## Step 8: Evaluate on the test split & compare

In [ ]:
# @title Evaluate best.pt of each model on the test split
for key, run in RUNS.items():
    ckpt = CKPT_ROOT / run["run"] / "best.pt"
    print(f"\n{'=' * 60}\n{run['label']}\n{'=' * 60}")
    if not ckpt.is_file():
        print(f"{ckpt} not found — skipping.")
        continue
    evaluate_checkpoint(CFGS[key], ckpt, split="test", max_batches=0,
                        save_json=CKPT_ROOT / run["run"] / "metrics_test.json")

In [ ]:
# @title Summary table
import pandas as pd

summary = []
for key, run in RUNS.items():
    path = CKPT_ROOT / run["run"] / "metrics_test.json"
    if path.is_file():
        data = json.loads(path.read_text(encoding="utf-8"))
        summary.append({
            "Experiment": run["label"],
            "Best epoch": data.get("epoch"),
            "Test mIoU (%)": f"{data.get('miou', 0) * 100:.2f}",
            "Mean Dice (%)": f"{data.get('mdice', 0) * 100:.2f}",
            "Pixel Acc (%)": f"{data.get('pixel_acc', 0) * 100:.2f}",
            "Test Loss": f"{data.get('eval_loss', 0):.4f}",
            "Eval Time (s)": f"{data.get('eval_seconds', 0):.1f}",
        })

if summary:
    display(pd.DataFrame(summary))
else:
    print("No metrics_test.json yet — run the evaluation cell above.")

## Step 9: Export results
You will be asked whether to **download** `log.csv`, `metrics_test.json`, `best.pt`, `last.pt` to your computer and/or **save them to Google Drive** (overwriting the files in `<folder>/checkpoints/...`). If you resumed from Drive, the same folder is offered by default.

In [ ]:
# @title Ask: download and/or save checkpoints & logs to Google Drive
import zipfile

all_files = sorted(p for p in CKPT_ROOT.rglob("*") if p.is_file() and not p.name.endswith(".tmp"))
if not all_files:
    print("Nothing to export — no checkpoints or logs were produced.")
else:
    print("Files available:")
    for p in all_files:
        print(f"   {str(p.relative_to(WORK_DIR)):70s} {p.stat().st_size / 1e6:8.1f} MB")

    print("\nWhat do you want to do with the checkpoints / logs / best.pt / last.pt?")
    print("   1 - Download to my computer")
    print("   2 - Save to Google Drive (overwrite)")
    print("   3 - Both")
    print("   0 - Nothing")
    choice = ""
    while choice not in ("0", "1", "2", "3"):
        choice = input("Your choice [0/1/2/3]: ").strip()

    if choice in ("2", "3"):
        target = None
        if DRIVE_FOLDER is not None and ask_yes_no(f"Use the same Drive folder as for resume ({DRIVE_FOLDER})?", default=True):
            target = DRIVE_FOLDER
        while target is None:
            link = input("Paste the Google Drive folder link (or a path like MyDrive/cityscapes_runs — created if missing): ")
            try:
                target = resolve_drive_folder(link, create=True)
            except Exception as e:
                print(f"{e}")
                if not ask_yes_no("Try another link?", default=True):
                    break
        if target is not None:
            dst_root = Path(target) / "checkpoints"
            print(f" Copying to {dst_root} (overwriting)...")
            for p in all_files:
                dst = dst_root / p.relative_to(CKPT_ROOT)
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(p, dst)
                print(f"   {dst}")
            DRIVE_FOLDER = target
            print("Saved to Google Drive. Next time, paste this folder link at the resume question.")

    if choice in ("1", "3"):
        from google.colab import files
        include_pt = ask_yes_no("Include the .pt weights (best.pt / last.pt, can be several hundred MB each)?", default=True)
        selected = [p for p in all_files if include_pt or p.suffix != ".pt"]
        zip_path = Path("/content/assignment2_checkpoints.zip")
        with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as zf:
            for p in selected:
                zf.write(p, p.relative_to(WORK_DIR))
        print(f"{zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB) — starting browser download...")
        print("   (For very large zips, the Drive option is more reliable than the browser download.)")
        files.download(str(zip_path))

    if choice == "0":
        print("Nothing exported. Files stay in", CKPT_ROOT, "until this Colab session ends.")